In [1]:
"""
NumPy Stock Market Analyzer
============================
Covers: array creation, slicing, fancy indexing, boolean masking,
        vectorization, broadcasting, axes, dtypes, and math functions.
"""

import numpy as np

# ── 1. SIMULATE STOCK DATA ──────────────────────────────────────────────────
# 5 stocks × 252 trading days (1 year), dtype=float32 to save memory
np.random.seed(42)
STOCKS = ["INFY", "HCLTECH", "ICICIBANK", "NTPC", "SUNPHARMA"]
DAYS = 252

# Random daily returns (mean ~0.05%, std ~1.5%)
daily_returns = np.random.normal(loc=0.0005, scale=0.015, size=(DAYS, len(STOCKS))).astype(np.float32)

# Build price series: start each stock at a base price
base_prices = np.array([1500, 1700, 1100, 380, 1800], dtype=np.float32)
prices = np.zeros((DAYS, len(STOCKS)), dtype=np.float32)
prices[0] = base_prices

# Vectorized price build (no loop needed for the math, loop only for dependency)
for d in range(1, DAYS):
    prices[d] = prices[d - 1] * (1 + daily_returns[d])  # broadcasting base × scalar

print("=" * 60)
print("  NumPy Stock Market Analyzer")
print("=" * 60)
print(f"\nData shape : {prices.shape}  → (days × stocks)")
print(f"dtype      : {prices.dtype}")
print(f"Memory     : {prices.nbytes:,} bytes  "
      f"(vs ~{prices.astype(np.float64).nbytes:,} bytes in float64)\n")


# ── 2. BASIC STATS (axis-aware) ──────────────────────────────────────────────
print("── Daily Return Stats (across all 252 days) ──")
for i, name in enumerate(STOCKS):
    col = daily_returns[:, i]          # slicing along axis 0
    print(f"  {name:<12} mean={np.mean(col)*100:+.3f}%  "
          f"std={np.std(col)*100:.3f}%  "
          f"max={np.max(col)*100:+.2f}%  "
          f"min={np.min(col)*100:+.2f}%")


# ── 3. CUMULATIVE RETURNS (vectorized, no loops) ─────────────────────────────
cum_returns = (prices[-1] - prices[0]) / prices[0] * 100   # broadcasting
print("\n── Annual Cumulative Return ──")
for i, name in enumerate(STOCKS):
    print(f"  {name:<12} {cum_returns[i]:+.2f}%")

best_idx  = np.argmax(cum_returns)
worst_idx = np.argmin(cum_returns)
print(f"\n  🏆 Best  performer : {STOCKS[best_idx]}  ({cum_returns[best_idx]:+.2f}%)")
print(f"  ⚠️  Worst performer : {STOCKS[worst_idx]}  ({cum_returns[worst_idx]:+.2f}%)")


# ── 4. VOLATILITY (30-day rolling std, using reshape trick) ──────────────────
# Trim to 240 days so it divides evenly into 8 windows of 30
prices_trim = prices[:240]                                  # slicing
windows = prices_trim.reshape(8, 30, len(STOCKS))           # reshape
window_returns = np.diff(windows, axis=1) / windows[:, :-1, :]
volatility = np.std(window_returns, axis=1) * np.sqrt(30)  # annualised-ish

print("\n── 30-day Volatility Windows (annualised) ──")
print(f"  {'Stock':<12}", *[f"W{i+1:02d}  " for i in range(8)])
for i, name in enumerate(STOCKS):
    vals = "  ".join(f"{v*100:.2f}%" for v in volatility[:, i])
    print(f"  {name:<12} {vals}")


# ── 5. BOOLEAN MASKING – flag crash days ──────────────────────────────────────
crash_threshold = -0.03   # any stock drops > 3% in a day
crash_mask = daily_returns < crash_threshold        # boolean array
crash_days, crash_stocks = np.where(crash_mask)    # fancy indexing equivalent

print(f"\n── Crash Events (single-day drop > 3%) ──")
print(f"  Total events: {len(crash_days)}")
for d, s in zip(crash_days[:5], crash_stocks[:5]):  # show first 5
    print(f"  Day {d:>3}  {STOCKS[s]:<12} return={daily_returns[d, s]*100:+.2f}%")
if len(crash_days) > 5:
    print(f"  ... and {len(crash_days)-5} more")


# ── 6. CORRELATION MATRIX (fancy indexing + corrcoef) ────────────────────────
corr = np.corrcoef(daily_returns.T)   # shape: (5, 5)
print("\n── Return Correlation Matrix ──")
header = f"  {'':12}" + "".join(f"{s:>10}" for s in STOCKS)
print(header)
for i, name in enumerate(STOCKS):
    row = "".join(f"{corr[i, j]:>10.3f}" for j in range(len(STOCKS)))
    print(f"  {name:<12}{row}")


# ── 7. NORMALISATION (broadcasting) ──────────────────────────────────────────
mean_p = np.mean(prices, axis=0)   # shape (5,) — mean per stock
std_p  = np.std(prices,  axis=0)
normalised = (prices - mean_p) / std_p   # broadcasting: (252,5) op (5,)

print(f"\n── Normalised Price Stats (should all be ≈0 mean, ≈1 std) ──")
for i, name in enumerate(STOCKS):
    print(f"  {name:<12} mean={np.mean(normalised[:,i]):+.4f}  std={np.std(normalised[:,i]):.4f}")


# ── 8. TOP-5 BEST SINGLE-DAY GAINS (fancy indexing) ──────────────────────────
flat_returns = daily_returns.flatten()
top5_idx = np.argpartition(flat_returns, -5)[-5:]          # fancy indexing
top5_idx = top5_idx[np.argsort(flat_returns[top5_idx])[::-1]]
print("\n── Top 5 Single-Day Gains ──")
for rank, idx in enumerate(top5_idx, 1):
    day, stock = divmod(idx, len(STOCKS))
    print(f"  #{rank}  Day {day:>3}  {STOCKS[stock]:<12} +{flat_returns[idx]*100:.2f}%")


print("\n" + "=" * 60)
print("  Analysis complete.")
print("=" * 60)

  NumPy Stock Market Analyzer

Data shape : (252, 5)  → (days × stocks)
dtype      : float32
Memory     : 5,040 bytes  (vs ~10,080 bytes in float64)

── Daily Return Stats (across all 252 days) ──
  INFY         mean=+0.072%  std=1.402%  max=+4.00%  min=-3.70%
  HCLTECH      mean=+0.003%  std=1.554%  max=+3.66%  min=-4.29%
  ICICIBANK    mean=+0.050%  std=1.449%  max=+3.89%  min=-4.81%
  NTPC         mean=+0.177%  std=1.454%  max=+4.67%  min=-3.93%
  SUNPHARMA    mean=+0.234%  std=1.541%  max=+5.83%  min=-3.88%

── Annual Cumulative Return ──
  INFY         +15.96%
  HCLTECH      -2.01%
  ICICIBANK    +9.26%
  NTPC         +48.44%
  SUNPHARMA    +75.67%

  🏆 Best  performer : SUNPHARMA  (+75.67%)
  ⚠️  Worst performer : HCLTECH  (-2.01%)

── 30-day Volatility Windows (annualised) ──
  Stock        W01   W02   W03   W04   W05   W06   W07   W08  
  INFY         8.18%  7.66%  6.47%  7.10%  7.48%  8.82%  7.79%  7.58%
  HCLTECH      7.37%  7.88%  8.81%  8.94%  8.61%  6.41%  8.57%  11.22%
  